In [1]:
import pandas as pd
import numpy as np
import os
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# NBA Game Prediction Model - XGBoost

This notebook implements an XGBoost model to predict NBA game outcomes based on team statistics.

In [2]:
# Read interim data
data_dir = os.path.join('..', 'data', 'processed')

train_data = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_data = pd.read_csv(os.path.join(data_dir, 'test.csv'))
validation_data = pd.read_csv(os.path.join(data_dir, 'validation.csv'))

In [3]:
def make_matchup_df(df):
    """Create simple feature matrix from games data"""
    features = []
    labels = []
    
    for gid, game in df.groupby('GAME_ID'):
        if len(game) != 2:
            continue
            
        team1 = game.iloc[0]
        team2 = game.iloc[1]
        
        # Get just the rolling features
        feature_cols = [col for col in df.columns if col.startswith('r50_')]
        
        # Calculate differences
        feature_vector = []
        for col in feature_cols:
            diff = float(team1[col]) - float(team2[col])
            feature_vector.append(diff)
            
        # Add home advantage
        feature_vector.append(1 if 'vs.' in team1['MATCHUP'] else 0)
        
        # Add to lists
        features.append(feature_vector)
        labels.append(1 if team1['WL'] == 'W' else 0)
            
    return np.array(features), np.array(labels)

In [4]:
# Create training dataset
X, y = make_matchup_df(train_data)

# Print dataset info
print(f"Dataset shape: {X.shape}")
print(f"Number of games: {len(X)}")
print(f"Number of features: {X.shape[1]}")
print(f"Class balance (wins/losses):", dict(zip(*np.unique(y, return_counts=True))))

Dataset shape: (11650, 9)
Number of games: 11650
Number of features: 9
Class balance (wins/losses): {np.int64(0): np.int64(5821), np.int64(1): np.int64(5829)}


In [5]:
# Train/test split and model training
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# XGBoost with optimized parameters for basketball prediction
clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_val)

# Print evaluation metrics
print('\nModel Evaluation:')
print(f'Training set size: {len(X_tr)} samples')
print(f'Validation set size: {len(X_val)} samples')
print(f'Accuracy on validation set: {accuracy_score(y_val, y_pred):.3f}')
print('\nClassification Report:')
print(classification_report(y_val, y_pred))


Model Evaluation:
Training set size: 9320 samples
Validation set size: 2330 samples
Accuracy on validation set: 0.637

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.64      0.64      1164
           1       0.64      0.64      0.64      1166

    accuracy                           0.64      2330
   macro avg       0.64      0.64      0.64      2330
weighted avg       0.64      0.64      0.64      2330



In [6]:
# Feature importance analysis
feature_cols = [col for col in train_data.columns if col.startswith('r50_')] + ['IS_HOME']
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': clf.feature_importances_
})
importance_df = importance_df.sort_values('importance', ascending=False)
print('\nTop 10 most important features:')
print(importance_df.head(10))


Top 10 most important features:
        feature  importance
8       IS_HOME    0.274359
4  r50_EffIndex    0.145873
7        r50_TS    0.097703
3    r50_TS_Pct    0.084614
2       r50_TSA    0.082988
5    r50_FG_Eff    0.080024
0  r50_MissedFG    0.079222
6  r50_RebRatio    0.079178
1  r50_MissedFT    0.076040


In [7]:
def predict_matchup(team1_abbr, team2_abbr, is_team1_home=True):
    """Simple prediction for a matchup between two teams using XGBoost"""
    # Get most recent stats for both teams
    team1_data = train_data[train_data['TEAM_ABBREVIATION'] == team1_abbr].iloc[-1]
    team2_data = train_data[train_data['TEAM_ABBREVIATION'] == team2_abbr].iloc[-1]
    
    # Just use the basic rolling stats and home advantage
    features_to_use = [col for col in train_data.columns if col.startswith('r50_')]
    
    # Build feature vector (differences between teams)
    feature_vector = []
    for col in features_to_use:
        diff = float(team1_data[col]) - float(team2_data[col])
        feature_vector.append(diff)
    
    # Add home court advantage
    feature_vector.append(1 if is_team1_home else 0)
    
    # Make prediction
    X_pred = np.array([feature_vector])
    prob = clf.predict_proba(X_pred)[0][1]
    pred = clf.predict(X_pred)[0]
    winner = team1_abbr if pred == 1 else team2_abbr
    

In [8]:
# Calculate team averages using the latest stats for each team
features_to_use = [col for col in train_data.columns if col.startswith('r50_')]
team_avgs = {}

for team in train_data['TEAM_ABBREVIATION'].unique():
    team_recent = train_data[train_data['TEAM_ABBREVIATION'] == team].iloc[-1]
    team_avgs[team] = team_recent[features_to_use]

team_avgs = pd.DataFrame(team_avgs).T 
print("XGBoost Team averages shape:", team_avgs.shape)
print("\nFirst few rows:")
print(team_avgs.head())

XGBoost Team averages shape: (32, 8)

First few rows:
    r50_MissedFG r50_MissedFT  r50_TSA r50_TS_Pct r50_EffIndex r50_FG_Eff  \
POR        50.16          4.2  101.316   56.08406       88.436   0.450623   
HOU        49.44         5.52  101.812  57.990289       91.436   0.449774   
BOS        47.24          4.3  98.5728  57.577617       96.246   0.462101   
MIA        45.14         5.26  95.5744  58.470106        93.18   0.462541   
UTA        45.08          5.3  95.7472   58.72241       88.912   0.469277   

    r50_RebRatio    r50_TS  
POR     0.223099  0.560841  
HOU     0.221566  0.579903  
BOS     0.219733  0.575776  
MIA     0.185901  0.584701  
UTA     0.195822  0.587224  


In [9]:
# Save XGBoost model and team averages for PredictionSimulator
joblib.dump(clf, 'xgb_model.pkl')
team_avgs.to_csv(os.path.join(data_dir, 'team_averages_xgb.csv'))